# 23 — Final fine-tuning — RT-DETRv2-L
Loads notebook 13's best configuration and runs the default headline matrix: the tuned recipe at seed 42 only. Set `FULL_MATRIX = True` to run the opt-in baseline+tuned × seeds 17/42/3407 variance matrix instead (six runs, reported separately).

In [ ]:
DATASET_TRACK = "2class"
START_FINETUNING = False
# False: the default headline matrix (tuned recipe, seed 42).
# True: the opt-in baseline+tuned x seeds 17/42/3407 variance
# matrix - six runs, reported separately from the headline table.
FULL_MATRIX = False
# False runs against session storage, which is DELETED when the
# session ends. Only for smoke runs - never HPO or final training.
USE_GOOGLE_DRIVE = True


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git"
REPOSITORY_BRANCH = "main"
SMOKE_TEST = os.environ.get("SMOKE_TEST", "").lower() in {"1", "true", "yes"}

# The only logic a notebook still owns: make `src` importable. Everything after
# this line - Git state, platform detection, paths, dependency policy - lives in
# src/notebook_bootstrap.py so all notebooks behave identically.
_override = os.environ.get("BENCHMARK_REPO_ROOT")
_candidates = (
    [Path(_override).expanduser()]
    if _override
    else [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/content/aerial-object-detection-benchmark"),
        Path("/kaggle/working/aerial-object-detection-benchmark"),
    ]
)
REPO_PATH = next(
    (
        candidate.resolve()
        for candidate in _candidates
        if (candidate / "src" / "notebook_bootstrap.py").is_file()
    ),
    None,
)
if REPO_PATH is None:
    _host = (
        Path("/content")
        if Path("/content").is_dir()
        else Path("/kaggle/working")
        if Path("/kaggle/working").is_dir()
        else None
    )
    if _host is None:
        raise RuntimeError(
            "Run this notebook from the repository, or set BENCHMARK_REPO_ROOT "
            "to an existing clone."
        )
    REPO_PATH = (_host / "aerial-object-detection-benchmark").resolve()
    REPO_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_PATH)],
        check=True,
    )
sys.path.insert(0, str(REPO_PATH))

from src.notebook_bootstrap import bootstrap_notebook

bootstrap = bootstrap_notebook(
    REPO_PATH,
    requirements_file=None,
    use_google_drive=USE_GOOGLE_DRIVE,
    smoke_test=SMOKE_TEST,
)
notebook_environment = bootstrap.environment
REPO_PATH = notebook_environment.repository_root
DRIVE_ROOT = notebook_environment.artifact_root
LOCAL_CACHE_ROOT = notebook_environment.local_cache_root
NOTEBOOK_PLATFORM = notebook_environment.platform
IN_COLAB = NOTEBOOK_PLATFORM == "colab"
IN_KAGGLE = NOTEBOOK_PLATFORM == "kaggle"
print(bootstrap.summary())


In [ ]:
MODEL_ID = "rtdetrv2_l"
from src.models.rtdetrv2.trainer import RTDetrSharedTrainer
print(f"Shared training engine: {RTDetrSharedTrainer.__name__}")
print("RT-DETR optimizer policy: rtdetr_recipe_v2")
if SMOKE_TEST:
    result = {"status": "guarded", "model_id": MODEL_ID}
else:
    from src.workflows.dataset_setup import require_prepared_dataset_track
    from src.workflows.environment import ensure_model_environment
    from src.hpo.final_workflow import FinalExperimentWorkflow
    require_prepared_dataset_track(DRIVE_ROOT, DATASET_TRACK)
    environment = (
        ensure_model_environment(MODEL_ID, REPO_PATH, DRIVE_ROOT)
        if START_FINETUNING
        else {"status": "SKIPPED_PREVIEW", "family": "rtdetr"}
    )
    result = FinalExperimentWorkflow(REPO_PATH, DRIVE_ROOT, MODEL_ID, DATASET_TRACK).run(
        start_expensive_stage=START_FINETUNING,
        full_matrix=FULL_MATRIX,
    )
result
